# NYC Yellow Taxi — Data Fetching Experiment
Fetch yellow taxi trip records for **2016-01-01 to 2016-01-07** using two approaches:
- **Approach A**: TLC direct parquet download (fast, official source)
- **Approach B**: NYC Open Data Socrata REST API (date-filtered, no full download needed)

In [17]:
import requests
import pandas as pd
import io

START_DATE = "2022-01-01"
END_DATE   = "2022-01-07"

## Approach A — TLC Direct Parquet Download
The TLC publishes monthly parquet files. Download January 2016 and filter to the target week.

URL pattern: `https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_YYYY-MM.parquet`

In [ ]:
# put as env
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2016-01.parquet"

print(f"Downloading: {url}")
response = requests.get(url)
response.raise_for_status()

df_month = pd.read_parquet(io.BytesIO(response.content))
print(f"Full month shape : {df_month.shape}")
print(f"Columns          : {df_month.columns.tolist()}")

In [ ]:
# Filter to 2016-01-01 - 2016-01-07 (inclusive)
mask = (
    (df_month["tpep_pickup_datetime"] >= START_DATE) &
    (df_month["tpep_pickup_datetime"] <  "2017-01-08")
)
df_week = df_month[mask].copy()

print(f"Week shape : {df_week.shape}")
df_week.head()

In [ ]:
df_week.describe()

## Approach B — NYC Open Data Socrata API
The NYC Open Data platform exposes a REST API (SODA) with server-side date filtering.

- Dataset: **2016 Yellow Taxi Trip Data** (ID: `biws-g3hs`)
- Results are paginated (max 50,000 per request); the loop handles all pages.

In [14]:
resp = requests.get(
"https://data.cityofnewyork.us/api/catalog/v1",
    params={"q": "yellow taxi trip data",
"limit": 30}
)

datasets = {
    r["resource"]["name"]: r["resource"]["id"]
    for r in resp.json()["results"]
    if "yellow taxi trip data" in
r["resource"]["name"].lower() and 
("2022" in r["resource"]["name"].lower() 
 or "2023" in r["resource"]["name"].lower())
}


In [15]:
datasets

{'2023 Yellow Taxi Trip Data': '4b4i-vvec',
 '2022 Yellow Taxi Trip Data': 'qp3b-zxtp'}

In [ ]:
# put into env or config
DATASET_IDs = datasets.values()
DATASET_ID = "qp3b-zxtp"
BASE_URL   = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.json"
PAGE_SIZE  = 50_000

# SoQL date filter — exclusive upper bound (< Jan 8)
where_clause = (
    f"tpep_pickup_datetime >= '{START_DATE}T00:00:00' "
    f"AND tpep_pickup_datetime < '2022-01-08T00:00:00'"
)

records = []
offset  = 0

while True:
    params = {
        "$where" : where_clause,
        "$limit" : PAGE_SIZE,
        "$offset": offset,
        "$order" : "tpep_pickup_datetime",
    }
    resp = requests.get(BASE_URL, params=params, timeout=60)
    resp.raise_for_status()

    page = resp.json()
    if not page:
        break

    records.extend(page)
    print(f"  Fetched {len(records):,} records so far (offset={offset})")

    if len(page) < PAGE_SIZE:
        break
    offset += PAGE_SIZE

df_api = pd.DataFrame(records)
print(f"Total records via API: {len(df_api):,}")

  Fetched 50,000 records so far (offset=0)
  Fetched 100,000 records so far (offset=50000)
  Fetched 150,000 records so far (offset=100000)
  Fetched 200,000 records so far (offset=150000)
  Fetched 250,000 records so far (offset=200000)
  Fetched 300,000 records so far (offset=250000)
  Fetched 350,000 records so far (offset=300000)
  Fetched 400,000 records so far (offset=350000)
  Fetched 450,000 records so far (offset=400000)


KeyboardInterrupt: 

In [7]:
df_api = pd.DataFrame(records)

In [16]:
df_api.describe()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
count,450000,450000,450000,450000,450000,450000,450000,450000,450000,450000,450000,450000,450000,450000,450000,450000,450000
unique,2,132390,133714,10,2791,7,2,237,257,4,517,7,3,1531,226,3,4050
top,2,2017-01-01T01:40:41.000,2017-01-02T00:00:00.000,1,1.00,1,N,79,236,1,6,0,0.5,0,0,0.3,6.8
freq,255159,19,42,297375,8989,432671,448751,16677,13894,253621,24105,277238,446732,210373,425236,449670,13436


In [ ]:
# Parse datetimes
df_api["tpep_pickup_datetime"]  = pd.to_datetime(df_api["tpep_pickup_datetime"])
df_api["tpep_dropoff_datetime"] = pd.to_datetime(df_api["tpep_dropoff_datetime"])

# Coerce numeric columns
num_cols = ["passenger_count", "trip_distance", "fare_amount",
            "tip_amount", "total_amount"]
for col in num_cols:
    if col in df_api.columns:
        df_api[col] = pd.to_numeric(df_api[col], errors="coerce")

df_api.head()

In [ ]:
# Daily trip counts
df_api["date"] = df_api["tpep_pickup_datetime"].dt.date
df_api.groupby("date").size().rename("trip_count")

## Batch Fetch → Parquet (memory-efficient)
Fetch one page at a time and write each batch as a row group into a single parquet file.  
RAM usage stays bounded to one batch (~50k rows) regardless of total data size.

In [ ]:
import os
import pyarrow as pa
import pyarrow.parquet as pq

# ── config ────────────────────────────────────────────────────────────────────
DATASET_ID       = "qp3b-zxtp"          # 2022 Yellow Taxi Trip Data
BASE_URL         = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.json"
PAGE_SIZE        = 50_000
START_DATE       = "2022-01-01"
END_DATE         = "2022-01-14"
END_EXCLUSIVE    = "2022-01-15"

# ── output path ───────────────────────────────────────────────────────────────
os.makedirs("data", exist_ok=True)
output_path = f"data/yellow_tripdata_{START_DATE}_{END_DATE}.parquet"

# ── dtypes applied to every batch ─────────────────────────────────────────────
DATETIME_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]
FLOAT_COLS    = ["trip_distance", "fare_amount", "extra", "mta_tax",
                 "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount"]
INT_COLS      = ["vendorid", "passenger_count", "ratecodeid",
                 "payment_type", "pulocationid", "dolocationid"]

def cast_batch(df: pd.DataFrame) -> pd.DataFrame:
    for col in DATETIME_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    for col in FLOAT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")
    for col in INT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int16")
    return df

# ── batch fetch + stream write ────────────────────────────────────────────────
where_clause = (
    f"tpep_pickup_datetime >= '{START_DATE}T00:00:00' "
    f"AND tpep_pickup_datetime < '{END_EXCLUSIVE}T00:00:00'"
)

writer      = None
offset      = 0
total_rows  = 0

try:
    while True:
        params = {
            "$where" : where_clause,
            "$limit" : PAGE_SIZE,
            "$offset": offset,
            "$order" : "tpep_pickup_datetime",
        }
        resp = requests.get(BASE_URL, params=params, timeout=60)
        resp.raise_for_status()

        page = resp.json()
        if not page:
            break

        batch = cast_batch(pd.DataFrame(page))
        table = pa.Table.from_pandas(batch, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema, compression="snappy")

        writer.write_table(table)
        total_rows += len(batch)
        print(f"  offset={offset:>7,}  batch={len(batch):,}  total={total_rows:,}")

        # done when page is smaller than requested
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

finally:
    if writer:
        writer.close()
        print(f"\nSaved → {output_path}  ({total_rows:,} rows)")

  offset=      0  batch=50,000  total=50,000
  offset= 50,000  batch=50,000  total=100,000
  offset=100,000  batch=50,000  total=150,000
  offset=150,000  batch=50,000  total=200,000
  offset=200,000  batch=50,000  total=250,000
  offset=250,000  batch=50,000  total=300,000
  offset=300,000  batch=50,000  total=350,000
  offset=350,000  batch=50,000  total=400,000
  offset=400,000  batch=50,000  total=450,000


In [ ]:
# Verify the saved file
saved = pq.read_metadata(output_path)
print(f"Row groups : {saved.num_row_groups}")
print(f"Total rows : {saved.num_rows:,}")
print(f"File size  : {os.path.getsize(output_path) / 1024**2:.1f} MB")

Row groups : 10
Total rows : 494,920
File size  : 10.2 MB


In [ ]:
df = pd.read_parquet(output_path)

In [ ]:
df

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2022-01-01 00:00:08,2022-01-01 00:14:14,1,7.94,1,N,138,255,1,23.0,0.5,0.5,4.86,0.00,0.3,30.410000,0.0,1.25
1,2,2022-01-01 00:00:11,2022-01-01 00:14:29,1,5.55,1,N,132,215,2,17.5,0.5,0.5,0.00,0.00,0.3,20.049999,0.0,1.25
2,2,2022-01-01 00:00:17,2022-01-01 00:31:04,1,3.86,1,N,170,68,1,20.0,0.5,0.5,4.76,0.00,0.3,28.559999,2.5,0.0
3,2,2022-01-01 00:00:18,2022-01-01 00:13:27,1,3.37,1,N,79,209,1,13.0,0.5,0.5,3.36,0.00,0.3,20.160000,2.5,0.0
4,2,2022-01-01 00:00:20,2022-01-01 00:20:51,1,4.89,1,N,68,256,1,18.0,0.5,0.5,6.54,0.00,0.3,28.340000,2.5,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494915,2,2022-01-07 23:59:52,2022-01-08 00:17:20,1,3.99,1,N,249,255,1,15.5,0.5,0.5,3.86,0.00,0.3,23.160000,2.5,0.0
494916,2,2022-01-07 23:59:53,2022-01-08 00:24:30,1,10.41,1,N,138,246,1,31.0,0.5,0.5,8.27,6.55,0.3,50.869999,2.5,1.25
494917,2,2022-01-07 23:59:53,2022-01-08 00:10:25,3,2.54,1,N,79,229,1,10.0,0.5,0.5,2.76,0.00,0.3,16.559999,2.5,0.0
494918,2,2022-01-07 23:59:55,2022-01-08 00:20:14,1,5.06,1,N,100,256,2,18.5,0.5,0.5,0.00,0.00,0.3,22.299999,2.5,0.0
